# AI-Powered Stock Price Prediction & Portfolio Management Using LSTM
**Academic Major B.Tech Data Science Project**
**Application Name**: StockMind AI
**Primary AI Architecture**: Stacked Long Short-Term Memory (LSTM) Neural Network

---
## Notebook Overview
This notebook demonstrates the end-to-end data science pipeline:
1. **Historical Market Data Loading**: Load OHLCV datasets for representative stocks (AAPL, MSFT, GOOGL, TSLA, NVDA).
2. **Feature Engineering**: Compute technical indicators (SMA 20/50, EMA 20, RSI 14, MACD, Volatility).
3. **Time-Series Preprocessing**: MinMaxScaler normalization and strict chronological 60-day window sequence generation (70% Train, 15% Val, 15% Test).
4. **Stacked LSTM Training**: Keras sequential model with EarlyStopping and ModelCheckpointing.
5. **Model Evaluation**: Evaluation on unseen test data using MAE, RMSE, MAPE (%), and R² Score.
6. **Rule-Based Signal & Portfolio Risk Analytics**: Sharpe Ratio, Volatility, Max Drawdown, and HHI Concentration.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
print('TensorFlow Version:', tf.__version__)

## 1. Data Loading & Inspection

In [ ]:
data_path = os.path.join('data', 'AAPL.csv')
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df.sort_index(inplace=True)
print(f'Loaded {len(df)} historical trading days for AAPL.')
df.head()

## 2. Technical Indicators Engineering

In [ ]:
# Simple & Exponential Moving Averages
df['SMA_20'] = df['Close'].rolling(window=20, min_periods=1).mean()
df['SMA_50'] = df['Close'].rolling(window=50, min_periods=1).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# RSI (14 Period)
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14, min_periods=1).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14, min_periods=1).mean()
rs = gain / (loss + 1e-10)
df['RSI_14'] = 100 - (100 / (1 + rs))

# MACD
ema12 = df['Close'].ewm(span=12, adjust=False).mean()
ema26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema12 - ema26
df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

df.bfill(inplace=True)
df.head()

## 3. Preprocessing & Chronological Train/Val/Test Split

In [ ]:
feature_cols = ['Close', 'Open', 'High', 'Low', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI_14', 'MACD']
features = df[feature_cols].values
target = df[['Close']].values

# 70% Train, 15% Validation, 15% Testing
n_total = len(df)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

scaler_x = MinMaxScaler()
scaler_y = MinMaxScaler()

# Fit ONLY on training set to prevent data leakage!
scaled_train_x = scaler_x.fit_transform(features[:n_train])
scaled_train_y = scaler_y.fit_transform(target[:n_train])

scaled_val_x = scaler_x.transform(features[n_train-60 : n_train+n_val])
scaled_val_y = scaler_y.transform(target[n_train-60 : n_train+n_val])

scaled_test_x = scaler_x.transform(features[n_train+n_val-60 :])
scaled_test_y = scaler_y.transform(target[n_train+n_val-60 :])

def create_sequences(X, y, lookback=60):
    seq_x, seq_y = [], []
    for i in range(lookback, len(X)):
        seq_x.append(X[i-lookback : i])
        seq_y.append(y[i, 0])
    return np.array(seq_x), np.array(seq_y)

X_train, y_train = create_sequences(scaled_train_x, scaled_train_y)
X_val, y_val = create_sequences(scaled_val_x, scaled_val_y)
X_test, y_test = create_sequences(scaled_test_x, scaled_test_y)

print(f'X_train shape: {X_train.shape}')
print(f'X_val shape:   {X_val.shape}')
print(f'X_test shape:  {X_test.shape}')

## 4. Stacked Keras LSTM Model Architecture

In [ ]:
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='huber', metrics=['mae'])
model.summary()

## 5. Model Evaluation Metrics

In [ ]:
# Predict on Test Set
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=10, batch_size=32, verbose=1)
scaled_preds = model.predict(X_test)
preds = scaler_y.inverse_transform(scaled_preds).flatten()
actuals = target[n_train+n_val : n_train+n_val+len(preds)].flatten()

mae = mean_absolute_error(actuals, preds)
rmse = np.sqrt(mean_squared_error(actuals, preds))
mape = np.mean(np.abs((actuals - preds) / actuals)) * 100
r2 = r2_score(actuals, preds)

print(f'=== LSTM Model Evaluation Metrics ===')
print(f'MAE:  ${mae:.2f}')
print(f'RMSE: ${rmse:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'R²:   {r2:.4f}')